# NB04 · Semana 8 — Transformación con dbt (silver → gold)

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Formalizar las transformaciones como **código** con dbt: pasar de bronze a **silver** (tipado y limpieza) y a **gold** (métricas para consumo), con **tests de calidad** automatizados.

> Base de la **Entrega 2 (E2)**. Este notebook orquesta comandos dbt; el proyecto dbt vive en la carpeta `dbt/` del repositorio.

## 1. Estructura del proyecto dbt

Revisamos los modelos provistos como punto de partida. Cada grupo los adapta a su dominio.

In [3]:
import subprocess, os
DBT_DIR = './dbt'
for root, _, files in os.walk(os.path.join(DBT_DIR, 'models')):
    for f in sorted(files):
        print(os.path.relpath(os.path.join(root, f), DBT_DIR))

models/fct_amenazas_mensual.sql
models/schema.yml
models/stg_urlhaus.sql
models/.ipynb_checkpoints/schema-checkpoint.yml
models/.ipynb_checkpoints/stg_urlhaus-checkpoint.sql


## 2. Cargar los seeds

`dbt seed` carga los CSV semilla como tablas de partida (bronze) dentro de DuckDB.

In [2]:
# TODO(alumno): ejecutar 'dbt seed' en DBT_DIR con --profiles-dir .
# usar subprocess.run y mostrar la salida.
...

#Omitimos dbt seed. Leemos directo desde la capa Bronze en MinIO (S3).

Ellipsis

In [4]:
# LIMPIADOR DE CACHE 
import shutil, os

rutas = [
    './dbt/models/.ipynb_checkpoints',
    './dbt/.ipynb_checkpoints'
]

for ruta in rutas:
    if os.path.exists(ruta):
        shutil.rmtree(ruta)
        print(f"Basura eliminada en: {ruta}")

Basura eliminada en: ./dbt/models/.ipynb_checkpoints
Basura eliminada en: ./dbt/.ipynb_checkpoints


In [5]:
!rm -rf ./dbt/models/.ipynb_checkpoints/

## 3. Construir modelos y correr tests

`dbt build` ejecuta modelos y tests en orden de dependencias. Los tests (`not_null`, `unique`) validan la calidad de la capa silver.

In [11]:
# TODO(alumno): ejecutar 'dbt build --profiles-dir .' y verificar que
# todos los tests queden en verde (PASS).

import subprocess

# Compila y ejecuta la transformación Silver y Gold, pasando los tests[cite: 3]
resultado_build = subprocess.run(
    ['dbt', 'build', '--profiles-dir', '.'], 
    cwd=DBT_DIR, 
    capture_output=True, 
    text=True
)
print(resultado_build.stdout)

00:45:27  Running with dbt=1.12.0
00:45:28  Registered adapter: duckdb=1.10.1
00:45:28  Found 2 models, 7 data tests, 486 macros
00:45:28  
00:45:28  Concurrency: 1 threads (target='dev')
00:45:28  
00:45:29  1 of 9 START sql table model main.stg_urlhaus .................................. [RUN]
00:45:29  1 of 9 OK created sql table model main.stg_urlhaus ............................. [OK in 0.24s]
00:45:29  2 of 9 START test not_null_stg_urlhaus_fecha_agregado .......................... [RUN]
00:45:29  2 of 9 PASS not_null_stg_urlhaus_fecha_agregado ................................ [PASS in 0.03s]
00:45:29  3 of 9 START test not_null_stg_urlhaus_id_amenaza .............................. [RUN]
00:45:29  3 of 9 PASS not_null_stg_urlhaus_id_amenaza .................................... [PASS in 0.01s]
00:45:29  4 of 9 START test unique_stg_urlhaus_id_amenaza ................................ [RUN]
00:45:29  4 of 9 PASS unique_stg_urlhaus_id_amenaza ...................................... [PA

## 4. Inspeccionar la capa gold

Consultamos el resultado final: métricas mensuales por variable, listas para serving/BI.

In [7]:
import duckdb
import os

DBT_DIR = './dbt'

# Conectamos a la base y consultamos la tabla física directamente
con = duckdb.connect(os.path.join(DBT_DIR, 'warehouse.duckdb'))

try:
    gold = con.sql("SELECT * FROM fct_amenazas_mensual LIMIT 10").df()
    print("--- Capa Gold (Métricas Mensuales de Malware) ---")
    print(gold)
except duckdb.Error as e:
    print(f"Error al consultar DuckDB: {e}")
finally:
    # Esta línea es la magia: cierra la conexión y libera el archivo
    # para que dbt pueda volver a correr sin tirar error de "lock"
    con.close()

--- Capa Gold (Métricas Mensuales de Malware) ---
   anio  mes      tipo_amenaza  cantidad_amenazas
0  2026   10  malware_download              28381
1  2026    9  malware_download              15204
2  2026    8  malware_download              15663
3  2026    7  malware_download               9630
4  2026    6  malware_download               4844
5  2026    5  malware_download                 82
6  2026    4  malware_download                320
7  2026    3  malware_download                248
8  2026    2  malware_download                153
9  2026    1  malware_download                148


## Entrega 2 (E2)

Lo que se evalúa: transformación bronze→silver→gold con dbt, tests de calidad en verde y documentación de modelos/linaje. Enunciado y rúbrica en la semana 7.

In [9]:
!rm /home/jovyan/work/dags/.env

rm: cannot remove '/home/jovyan/work/dags/.env': No such file or directory
